# 日频本地 CPU 完整训练

本 Notebook 在本机依次运行数据预处理、GFlowNet、AlphaEval 和 LightGBM。RQAlphaPlus 为可选阶段，只有本机已获得授权并准备好 bundle 时才启用。所有参数来自 `configs/daily/local.yaml`。

In [ ]:
from pathlib import Path
import os

search_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in search_roots if (p / 'pyproject.toml').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('请从 AlphaMining-GFlowNet-AlphaEval 仓库内启动 JupyterLab')
os.chdir(PROJECT_ROOT)
print('project root =', PROJECT_ROOT)

## 1. 安装依赖

首次运行或环境变化后执行。RQAlphaPlus 不会通过公开依赖安装。

In [ ]:
%pip install -q -r requirements.txt

## 2. 检查本机与配置

In [ ]:
import os
import platform
import sys
import torch
import yaml

CONFIG_PATH = Path('configs/daily/local.yaml')
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
print('Python =', sys.version.split()[0])
print('Platform =', platform.platform())
print('Logical CPUs =', os.cpu_count())
print('PyTorch =', torch.__version__)
print('CUDA visible before launcher =', torch.cuda.is_available())
print('Config =', CONFIG_PATH)
print('Date split =', config['dataset']['mining_start_date'], '->', config['dataset']['mining_end_date'], '/', config['dataset']['out_of_sample_start_date'], '->', config['dataset']['out_of_sample_end_date'])
price_path = Path(config['dataset']['file'])
if not price_path.exists():
    raise FileNotFoundError(f'缺少本地行情文件：{price_path.resolve()}')
print('price.csv MB =', round(price_path.stat().st_size / 1024**2, 1))

## 3. 选择运行范围

首次完整运行保持默认值。中断后可把 `FROM_STAGE` 改为 `alpha_eval` 或 `lightgbm`；如已有同配置的因子池，可启用 `REUSE_ALPHA_POOL` 只重算因子值。

In [ ]:
FROM_STAGE = 'prepare'       # prepare / gflownet / alpha_eval / lightgbm / backtest
TO_STAGE = 'lightgbm'        # 本地未安装RQAlphaPlus时不要设为backtest
REUSE_PREPARED_DATA = False # data/daily_price.pkl确认与当前config一致时可设True
REUSE_ALPHA_POOL = False     # 复用results/daily_local/alpha_pool.csv
POOL_SIZE = None             # None表示读取config；测试可设5
TORCH_THREADS = None         # None表示读取config；也可设为物理核心附近的数值
RQALPHA_BUNDLE = None        # 例如 Path.home() / '.rqalpha-plus/bundle'

## 4. 启动独立训练进程

使用独立进程是为了在导入 NumPy、PyTorch 前固定 CPU 和 BLAS 线程。终端进度会同时写入 `results/daily_local/logs/`。

In [ ]:
import subprocess

command = [
    sys.executable, 'scripts/train_daily_local.py',
    '--config', str(CONFIG_PATH),
    '--from-stage', FROM_STAGE,
    '--to-stage', TO_STAGE,
]
if REUSE_PREPARED_DATA:
    command.append('--reuse-prepared-data')
if REUSE_ALPHA_POOL:
    command.append('--reuse-alpha-pool')
if POOL_SIZE is not None:
    command.extend(['--pool-size', str(POOL_SIZE)])
if TORCH_THREADS is not None:
    command.extend(['--threads', str(TORCH_THREADS)])
if RQALPHA_BUNDLE is not None:
    command.extend(['--rqalpha-bundle', str(RQALPHA_BUNDLE)])
print(' '.join(command))
subprocess.run(command, check=True)

## 5. 查看阶段状态与产物

In [ ]:
import json

manifest_path = Path(config['outputs']['pipeline_manifest'])
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
print(json.dumps(manifest, ensure_ascii=False, indent=2))
for key in ('checkpoint', 'alpha_pool', 'factor_matrix', 'oos_factor_matrix', 'alpha_eval'):
    path = Path(config['outputs'][key])
    print(f'{key:20s}', path.exists(), path)
prediction = Path(config['outputs']['lightgbm_dir']) / 'prediction_score.csv'
print(f'{"prediction_score":20s}', prediction.exists(), prediction)